# 30 秒看懂：目标泄漏如何制造「高性能假象」

本 Notebook 用**本仓库真实的银行数据与真实的 DML 代码**，现场演示一件事：

> 一个衍生特征只要**间接使用了结果变量**，就会让 ATE 估计的偏差骤降、方差塌缩——
> 看起来像方法学突破，实际上是把答案抄给了模型。

**三种配置，其余完全相同，唯一区别是 `f_wood` 是否访问了 $Y(0)$：**

| 配置 | `f_wood` 的来源 |
|---|---|
| A 原始特征 | （不使用衍生特征） |
| B 衍生特征（含泄漏） | 由 `Total_Trans_Amt`（= $Y(0)$）构造 |
| C 衍生特征（修正） | 只用 `Total_Trans_Ct / Months_on_book`，不触碰结果变量 |

论文与完整实验见 `paper.md` 与 `src/paper_experiments.py`。

> **运行方式**：从仓库根目录执行 `jupyter notebook notebooks/`，或在 notebooks/ 目录下直接运行本文件。

In [ ]:
# 1) 环境：自动定位仓库根目录，加载本仓库自己的模块（不是玩具数据）
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def find_root():
    """从当前目录向上查找包含 src/dgp.py 的目录，避免硬编码相对路径。"""
    p = os.getcwd()
    for _ in range(5):
        if os.path.isfile(os.path.join(p, 'src', 'dgp.py')):
            return p
        p = os.path.dirname(p)
    raise RuntimeError('找不到仓库根目录（需要存在 src/dgp.py）')

ROOT = find_root()
sys.path.insert(0, ROOT)
os.chdir(ROOT)                     # 之后统一用相对仓库根目录的路径

from src.dgp import load_data, simulate_treatment, simulate_outcome, true_ate_full, OUTCOME_BASE
from src.features import build_features
from src.dml import dml_estimate

print('仓库根目录:', ROOT)

In [ ]:
# 2) 数据与真值：真实协变量 + 已知因果效应的半合成 DGP
df = load_data('data/BankChurners.csv')
X_all = df.drop(columns=[OUTCOME_BASE])

rng = np.random.RandomState(0)
D   = simulate_treatment(df, rng)          # 处理分配（依赖 Credit_Limit / Months_Inactive）
y   = simulate_outcome(df, D)              # Y = Y(0) * 1.15^D
tau = true_ate_full(df[OUTCOME_BASE])      # 正确真值：0.15 * E[Y(0)] 全样本

print(f'样本量 n = {len(df)}')
print(f'真实 ATE  tau = {tau:.2f}')

In [ ]:
# 3) 关键的一行：泄漏开关
#    alpha=1 -> f_wood 由 Y(0) 构造（等价于把答案抄给模型）
#    alpha=0 -> f_wood 只用 Total_Trans_Ct / Months_on_book，不触碰结果变量
Xtr, _, Dtr, _, ytr, _ = train_test_split(X_all, D, y, test_size=0.2, random_state=0)
y0_tr = df.loc[Xtr.index, OUTCOME_BASE]    # Y(0)，只在这一处被使用

X_leaky = pd.concat([Xtr, build_features(Xtr, y0_tr, alpha=1.0)], axis=1)
X_clean = pd.concat([Xtr, build_features(Xtr, y0_tr, alpha=0.0)], axis=1)

print('两种配置的特征矩阵维度：', X_leaky.shape, X_clean.shape)
print('唯一区别：f_wood 是否访问了 Y(0)。其余 5 个衍生特征完全相同。')

In [ ]:
# 4) 单次运行：泄漏让偏差「变好」
a_o, s_o = dml_estimate(Xtr,     Dtr, ytr)
a_l, s_l = dml_estimate(X_leaky, Dtr, ytr)
a_c, s_c = dml_estimate(X_clean, Dtr, ytr)

print(f'真实 ATE tau = {tau:.2f}\n')
print(f'{"配置":<20}{"ATE 估计":>12}{"|偏差|":>10}{"报告 SE":>10}')
for name, a, s in [('A 原始特征', a_o, s_o),
                   ('B 衍生(含泄漏)', a_l, s_l),
                   ('C 衍生(修正)', a_c, s_c)]:
    print(f'{name:<20}{a:>12.2f}{abs(a-tau):>10.2f}{s:>10.2f}')

In [ ]:
# 5) 30 次独立随机种子：看分布，不看单次
rows = []
for seed in range(30):
    rng = np.random.RandomState(seed)
    D   = simulate_treatment(df, rng)
    y   = simulate_outcome(df, D)
    tau = true_ate_full(df[OUTCOME_BASE])
    Xtr, _, Dtr, _, ytr, _ = train_test_split(X_all, D, y, test_size=0.2, random_state=seed)
    y0_tr = df.loc[Xtr.index, OUTCOME_BASE]

    X_l = pd.concat([Xtr, build_features(Xtr, y0_tr, 1.0)], axis=1)
    X_c = pd.concat([Xtr, build_features(Xtr, y0_tr, 0.0)], axis=1)

    ao, _ = dml_estimate(Xtr,     Dtr, ytr)
    al, _ = dml_estimate(X_l,     Dtr, ytr)
    ac, _ = dml_estimate(X_c,     Dtr, ytr)
    rows.append(dict(ate_o=ao, ate_l=al, ate_c=ac,
                     err_o=abs(ao-tau), err_l=abs(al-tau), err_c=abs(ac-tau)))

res = pd.DataFrame(rows)
print('30 次独立种子完成')
print(f'处理组基线比对照组高 11.45%，所以对照组均值不能代替全样本均值。')

In [ ]:
# 6) 汇总：泄漏的两个指纹 —— 偏差下降 + 方差塌缩
summ = pd.DataFrame({
    '平均|偏差|':    [res.err_o.mean(), res.err_l.mean(), res.err_c.mean()],
    'ATE 估计 SD':   [res.ate_o.std(),  res.ate_l.std(),  res.ate_c.std()],
    '胜出次数(/30)': ['—',
                      f'{int((res.err_l < res.err_o).sum())}/30',
                      f'{int((res.err_c < res.err_o).sum())}/30'],
}, index=['A 原始特征', 'B 衍生(含泄漏)', 'C 衍生(修正)'])

print(summ.round(2).to_string())
collapse = res.ate_o.std() / res.ate_l.std()
print(f'\n方差塌缩倍数（A 的 SD / B 的 SD）= {collapse:.2f}x')
print('真方法不会让方差无故塌缩 4 倍 —— 这是泄漏的典型指纹。')

In [ ]:
# 7) 可视化：左图看偏差，右图看方差塌缩
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False   # 修复负号显示
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

bp = axes[0].boxplot([res.err_o, res.err_l, res.err_c],
                     tick_labels=['A 原始', 'B 含泄漏', 'C 修正'],
                     patch_artist=True, widths=.55,
                     medianprops=dict(color='k', lw=1.4))
for patch, c in zip(bp['boxes'], ['#4C72B0', '#C44E52', '#55A868']):
    patch.set_facecolor(c); patch.set_alpha(.6); patch.set_edgecolor('k')
axes[0].axhline(res.err_o.mean(), ls='--', lw=1, color='#4C72B0', alpha=.6)
axes[0].set_ylabel('|ATE 偏差|')
axes[0].set_title(f'30 次独立种子的 ATE 偏差分布（真值 tau={tau:.0f}）')
axes[0].grid(axis='y', ls=':', alpha=.6)

labels = ['A 原始', 'B 含泄漏', 'C 修正']
sds    = [res.ate_o.std(), res.ate_l.std(), res.ate_c.std()]
bars   = axes[1].bar(labels, sds, color=['#4C72B0', '#C44E52', '#55A868'], alpha=.65)
for b, v in zip(bars, sds):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.6, f'{v:.1f}', ha='center', fontsize=10)
axes[1].set_ylabel('ATE 估计的标准差（跨 30 次种子）')
axes[1].set_title(f'方差塌缩 {collapse:.2f}x —— 泄漏的指纹')
axes[1].grid(axis='y', ls=':', alpha=.6)

plt.tight_layout()
plt.show()

## 结论

**看左图**：含泄漏的配置（红）偏差中位数最低、箱体最窄——单看这张图，你会以为发现了方法学突破。

**看右图**：它的 ATE 估计标准差只有其他两组的 **1/4**。一个估计量如果真的变好了，方差可以降低；
但**降低 4 倍、且点估计均值几乎不动**，说明第一阶段的残差里几乎不剩任何「未解释的变异」——
也就是模型把答案背下来了。

**去掉泄漏之后（绿）**，与原始特征（蓝）几乎完全重合：
偏差变化 −0.65%，配对 t 检验 **p = 0.892**，30 次里赢 16 次——就是抛硬币。

---

**给面试官的一句话**：

> 这个 Notebook 演示的是我在自己项目里犯过的错误。它让我明白：在因果推断里，
> 最危险的不是模型不够强，而是特征构造中一条看不见的信息通道；
> 以及——**验证如果和被验证对象共享同一个错误前提，它只会确认，不会质询。**